In [1]:
!pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sqlite3
import requests
import pandas as pd

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import display
print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
os.makedirs("data_pipeline", exist_ok=True)
os.makedirs("data_pipeline/data", exist_ok=True)
os.makedirs("data_pipeline/output", exist_ok=True)
print("Project folders created successfully!")


Project folders created successfully!


In [4]:
BASE_URL = "https://books.toscrape.com/"

START_URL = urljoin(
    BASE_URL,
    "catalogue/category/books_1/index.html"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Educational Data Pipeline Project)"
}

print("Website URL:")
print(START_URL)

Website URL:
https://books.toscrape.com/catalogue/category/books_1/index.html


In [5]:
try:
    response = requests.get(
        START_URL,
        headers=HEADERS,
        timeout=20
    )
    response.raise_for_status()
    print("website connection successful!")
    print("status code:", response.status_code)
except requests.RequestException as e:
    print("website connection failed!")
    print(e)

website connection successful!
status code: 200


In [6]:
def get_category_links():

    response = requests.get(
        START_URL,
        headers=HEADERS,
        timeout=20
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    categories = []

    category_section = soup.select_one(
        ".side_categories ul li ul"
    )

    if category_section is None:
        raise RuntimeError(
            "Could not find the category section."
        )

    for link in category_section.select("a"):

        category_name = link.get_text(
            strip=True
        )

        category_url = urljoin(
            START_URL,
            link.get("href")
        )

        categories.append({
            "category": category_name,
            "url": category_url
        })

    return categories

In [7]:
categories = get_category_links()

print(
    "Number of categories found:",
    len(categories)
)

print("\nFirst 10 categories:")

for category in categories[:10]:
    print(category)

Number of categories found: 50

First 10 categories:
{'category': 'Travel', 'url': 'https://books.toscrape.com/catalogue/category/books/travel_2/index.html'}
{'category': 'Mystery', 'url': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html'}
{'category': 'Historical Fiction', 'url': 'https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html'}
{'category': 'Sequential Art', 'url': 'https://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html'}
{'category': 'Classics', 'url': 'https://books.toscrape.com/catalogue/category/books/classics_6/index.html'}
{'category': 'Philosophy', 'url': 'https://books.toscrape.com/catalogue/category/books/philosophy_7/index.html'}
{'category': 'Romance', 'url': 'https://books.toscrape.com/catalogue/category/books/romance_8/index.html'}
{'category': 'Womens Fiction', 'url': 'https://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html'}
{'category': 'Fiction', 'url': 'htt

In [8]:
def scrape_category(category_name, category_url):

    books = []

    current_url = category_url

    while current_url:

        print(
            f"Scraping {category_name}: "
            f"{current_url}"
        )

        try:
            response = requests.get(
                current_url,
                headers=HEADERS,
                timeout=20
            )

            response.raise_for_status()

        except requests.RequestException as e:

            print(
                f"Request failed for "
                f"{current_url}: {e}"
            )

            break

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        product_cards = soup.select(
            "article.product_pod"
        )

        for book in product_cards:

            title_tag = book.select_one(
                "h3 a"
            )

            price_tag = book.select_one(
                ".price_color"
            )

            rating_tag = book.select_one(
                "p.star-rating"
            )

            availability_tag = book.select_one(
                ".availability"
            )

            # Title
            title = None

            if title_tag:
                title = title_tag.get(
                    "title",
                    ""
                ).strip()

            # Price
            price = None

            if price_tag:
                price = price_tag.get_text(
                    strip=True
                )

            # Availability
            availability = None

            if availability_tag:
                availability = (
                    availability_tag
                    .get_text(
                        " ",
                        strip=True
                    )
                )

            # Star rating
            star_rating = None

            if rating_tag:

                rating_classes = (
                    rating_tag.get(
                        "class",
                        []
                    )
                )

                for value in [
                    "One",
                    "Two",
                    "Three",
                    "Four",
                    "Five"
                ]:

                    if value in rating_classes:
                        star_rating = value
                        break

            books.append({
                "title": title,
                "price": price,
                "star_rating": star_rating,
                "availability": availability,
                "category": category_name
            })

        # Check pagination
        next_button = soup.select_one(
            "li.next a"
        )

        if next_button:

            current_url = urljoin(
                current_url,
                next_button.get("href")
            )

        else:

            current_url = None

    return books

In [9]:
all_books = []

used_categories = []

for category in categories:

    category_books = scrape_category(
        category["category"],
        category["url"]
    )

    if len(category_books) == 0:
        continue

    all_books.extend(
        category_books
    )

    used_categories.append(
        category["category"]
    )

    print(
        f"\nCompleted: "
        f"{category['category']}"
    )

    print(
        "Books in category:",
        len(category_books)
    )

    print(
        "Total books:",
        len(all_books)
    )

    print("-" * 50)

    # Assignment requirement
    if (
        len(all_books) >= 60
        and len(used_categories) >= 3
    ):
        break


print("\nSCRAPING COMPLETED")

print(
    "Total books:",
    len(all_books)
)

print(
    "Total categories:",
    len(used_categories)
)

print(
    "Categories:",
    used_categories
)

Scraping Travel: https://books.toscrape.com/catalogue/category/books/travel_2/index.html

Completed: Travel
Books in category: 11
Total books: 11
--------------------------------------------------
Scraping Mystery: https://books.toscrape.com/catalogue/category/books/mystery_3/index.html
Scraping Mystery: https://books.toscrape.com/catalogue/category/books/mystery_3/page-2.html

Completed: Mystery
Books in category: 32
Total books: 43
--------------------------------------------------
Scraping Historical Fiction: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html
Scraping Historical Fiction: https://books.toscrape.com/catalogue/category/books/historical-fiction_4/page-2.html

Completed: Historical Fiction
Books in category: 26
Total books: 69
--------------------------------------------------

SCRAPING COMPLETED
Total books: 69
Total categories: 3
Categories: ['Travel', 'Mystery', 'Historical Fiction']


In [10]:
raw_df = pd.DataFrame(
    all_books
)

print(
    "Dataset shape:",
    raw_df.shape
)

display(
    raw_df.head(10)
)

Dataset shape: (69, 5)


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel
5,A Summer In Europe,Â£44.34,Two,In stock,Travel
6,The Great Railway Bazaar,Â£30.54,One,In stock,Travel
7,A Year in Provence (Provence #1),Â£56.88,Four,In stock,Travel
8,The Road to Little Dribbling: Adventures of an...,Â£23.21,One,In stock,Travel
9,Neither Here nor There: Travels in Europe,Â£38.95,Three,In stock,Travel


In [11]:
print(
    raw_df["category"]
    .value_counts()
)

category
Mystery               32
Historical Fiction    26
Travel                11
Name: count, dtype: int64


In [12]:
print("Missing values:")

print(
    raw_df.isnull().sum()
)

Missing values:
title           0
price           0
star_rating     0
availability    0
category        0
dtype: int64


In [13]:
raw_df.to_csv(
    "data_pipeline/data/raw_books.csv",
    index=False
)

print(
    "Raw dataset saved successfully!"
)

Raw dataset saved successfully!


In [14]:
clean_df = raw_df.copy()

print(
    "Rows before cleaning:",
    len(clean_df)
)

Rows before cleaning: 69


In [15]:
clean_df["price_gbp"] = (
    clean_df["price"]
    .astype(str)
    .str.replace(
        "£",
        "",
        regex=False
    )
    .str.replace(
        "Â",
        "",
        regex=False
    )
)

clean_df["price_gbp"] = pd.to_numeric(
    clean_df["price_gbp"],
    errors="coerce"
)

display(
    clean_df[
        ["price", "price_gbp"]
    ].head(10)
)

,price,price_gbp
0,Â£45.17,45.17
1,Â£49.43,49.43
2,Â£48.87,48.87
3,Â£36.94,36.94
4,Â£37.33,37.33
5,Â£44.34,44.34
6,Â£30.54,30.54
7,Â£56.88,56.88
8,Â£23.21,23.21
9,Â£38.95,38.95


In [16]:
RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

clean_df["rating"] = (
    clean_df["star_rating"]
    .map(RATING_MAP)
)

clean_df["rating"] = pd.to_numeric(
    clean_df["rating"],
    errors="coerce"
)

display(
    clean_df[
        ["star_rating", "rating"]
    ].head(10)
)

,star_rating,rating
0,Two,2
1,Four,4
2,Three,3
3,Two,2
4,Three,3
5,Two,2
6,One,1
7,Four,4
8,One,1
9,Three,3


In [18]:
clean_df["in_stock"] = (
    clean_df["availability"]
    .astype(str)
    .str.lower()
    .str.contains(
        "in stock",
        na=False
    )
)

display(
    clean_df[
        ["availability", "in_stock"]
    ].head(14)
)

,availability,in_stock
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True
5,In stock,True
6,In stock,True
7,In stock,True
8,In stock,True
9,In stock,True


In [19]:
print(
    "Missing price_gbp:",
    clean_df["price_gbp"]
    .isna()
    .sum()
)

print(
    "Missing rating:",
    clean_df["rating"]
    .isna()
    .sum()
)

print(
    "Missing title:",
    clean_df["title"]
    .isna()
    .sum()
)

print(
    "Missing category:",
    clean_df["category"]
    .isna()
    .sum()
)

Missing price_gbp: 0
Missing rating: 0
Missing title: 0
Missing category: 0


In [20]:
price_median = (
    clean_df["price_gbp"]
    .median()
)

rating_median = (
    clean_df["rating"]
    .median()
)

print(
    "Price median:",
    price_median
)

print(
    "Rating median:",
    rating_median
)

Price median: 30.54
Rating median: 3.0


In [21]:
clean_df["price_gbp"] = (
    clean_df["price_gbp"]
    .fillna(price_median)
)

clean_df["rating"] = (
    clean_df["rating"]
    .fillna(rating_median)
)

clean_df["rating"] = (
    clean_df["rating"]
    .round()
    .clip(1, 5)
    .astype(int)
)

print(
    "Numeric missing values handled."
)

Numeric missing values handled.


In [22]:
before_drop = len(
    clean_df
)

clean_df = (
    clean_df
    .dropna(
        subset=[
            "title",
            "category"
        ]
    )
    .copy()
)

after_drop = len(
    clean_df
)

print(
    "Rows before:",
    before_drop
)

print(
    "Rows after:",
    after_drop
)

print(
    "Rows dropped:",
    before_drop - after_drop
)

Rows before: 69
Rows after: 69
Rows dropped: 0


In [23]:
GBP_TO_INR = 105.50

clean_df["price_inr"] = (
    clean_df["price_gbp"]
    * GBP_TO_INR
).round(2)

display(
    clean_df[
        [
            "title",
            "price_gbp",
            "price_inr"
        ]
    ].head(10)
)

,title,price_gbp,price_inr
0,It's Only the Himalayas,45.17,4765.44
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86
2,See America: A Celebration of Our National Par...,48.87,5155.78
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17
4,Under the Tuscan Sun,37.33,3938.31
5,A Summer In Europe,44.34,4677.87
6,The Great Railway Bazaar,30.54,3221.97
7,A Year in Provence (Provence #1),56.88,6000.84
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.66
9,Neither Here nor There: Travels in Europe,38.95,4109.23


In [24]:
final_df = clean_df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
].copy()

display(
    final_df.head(10)
)

,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.44,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.78,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,True,Travel
5,A Summer In Europe,44.34,4677.87,2,True,Travel
6,The Great Railway Bazaar,30.54,3221.97,1,True,Travel
7,A Year in Provence (Provence #1),56.88,6000.84,4,True,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,True,Travel
9,Neither Here nor There: Travels in Europe,38.95,4109.23,3,True,Travel


In [25]:
print(
    final_df.dtypes
)

title            str
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category         str
dtype: object


In [26]:
print(
    "Total books:",
    len(final_df)
)

print(
    "Unique categories:",
    final_df["category"]
    .nunique()
)

print(
    "Ratings:",
    sorted(
        final_df["rating"]
        .unique()
    )
)

print(
    "In-stock values:",
    final_df["in_stock"]
    .unique()
)

Total books: 69
Unique categories: 3
Ratings: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
In-stock values: [ True]


In [27]:
assert len(final_df) >= 60, (
    "Dataset must contain at least 60 books."
)

assert (
    final_df["category"].nunique()
    >= 3
), "Need at least 3 categories."

assert (
    final_df["rating"]
    .between(1, 5)
    .all()
), "Ratings must be between 1 and 5."

print(
    "Basic acceptance checks passed!"
)

Basic acceptance checks passed!


In [28]:
final_df.to_csv(
    "data_pipeline/data/cleaned_books.csv",
    index=False
)

print(
    "Cleaned dataset saved successfully!"
)

Cleaned dataset saved successfully!


In [29]:
DB_PATH = "data_pipeline/books.db"

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(
    DB_PATH
)

conn.execute(
    "PRAGMA foreign_keys = ON"
)

cursor = conn.cursor()

print(
    "SQLite database created!"
)

SQLite database created!


In [30]:
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT NOT NULL UNIQUE
)
""")

conn.commit()

print(
    "Categories table created!"
)

Categories table created!


In [31]:
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,

    title TEXT NOT NULL,

    price_gbp REAL NOT NULL,

    price_inr REAL NOT NULL,

    rating INTEGER NOT NULL
        CHECK(rating BETWEEN 1 AND 5),

    in_stock INTEGER NOT NULL
        CHECK(in_stock IN (0, 1)),

    category_id INTEGER NOT NULL,

    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
)
""")

conn.commit()

print(
    "Books table created!"
)

Books table created!


In [32]:
unique_categories = sorted(
    final_df["category"]
    .dropna()
    .unique()
)

cursor.executemany(
    """
    INSERT INTO categories (
        category_name
    )
    VALUES (?)
    """,
    [
        (category,)
        for category
        in unique_categories
    ]
)

conn.commit()

print(
    "Categories inserted:",
    len(unique_categories)
)

Categories inserted: 3


In [33]:
categories_db_df = pd.read_sql(
    """
    SELECT *
    FROM categories
    ORDER BY category_id
    """,
    conn
)

display(
    categories_db_df
)

,category_id,category_name
0,1,Historical Fiction
1,2,Mystery
2,3,Travel


In [34]:
books_to_insert = pd.merge(
    final_df,
    categories_db_df,
    left_on="category",
    right_on="category_name",
    how="left"
)

display(
    books_to_insert.head(10)
)

,title,price_gbp,price_inr,rating,in_stock,category,category_id,category_name
0,It's Only the Himalayas,45.17,4765.44,2,True,Travel,3,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,True,Travel,3,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.78,3,True,Travel,3,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,True,Travel,3,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,True,Travel,3,Travel
5,A Summer In Europe,44.34,4677.87,2,True,Travel,3,Travel
6,The Great Railway Bazaar,30.54,3221.97,1,True,Travel,3,Travel
7,A Year in Provence (Provence #1),56.88,6000.84,4,True,Travel,3,Travel
8,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,True,Travel,3,Travel
9,Neither Here nor There: Travels in Europe,38.95,4109.23,3,True,Travel,3,Travel


In [35]:
book_records = []

for _, row in books_to_insert.iterrows():

    record = (
        row["title"],
        float(row["price_gbp"]),
        float(row["price_inr"]),
        int(row["rating"]),
        int(row["in_stock"]),
        int(row["category_id"])
    )

    book_records.append(
        record
    )

print(
    "Book records prepared:",
    len(book_records)
)

Book records prepared: 69


In [36]:
cursor.executemany(
    """
    INSERT INTO books (
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    )
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    book_records
)

conn.commit()

print(
    "Books inserted successfully!"
)

Books inserted successfully!


In [37]:
book_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM books
    """
).fetchone()[0]

category_count = cursor.execute(
    """
    SELECT COUNT(*)
    FROM categories
    """
).fetchone()[0]

print(
    "Books in database:",
    book_count
)

print(
    "Categories in database:",
    category_count
)

Books in database: 69
Categories in database: 3


In [38]:
query1 = """
SELECT
    book_id,
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp > 40
ORDER BY price_gbp DESC;
"""

query1_df = pd.read_sql(
    query1,
    conn
)

display(
    query1_df
)

,book_id,title,price_gbp,price_inr,rating
0,26,Boar Island (Anna Pigeon #19),59.48,6275.14,3
1,39,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4
2,8,A Year in Provence (Provence #1),56.88,6000.84,4
3,14,The Past Never Ends,56.50,5960.75,4
4,61,The Last Painting of Sara de Vos,55.55,5860.52,2
5,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5
6,23,Murder at the 42nd Street Library (Raymond Amb...,54.36,5734.98,4
7,17,The Last Mile (Amos Decker #2),54.21,5719.16,2
8,43,1st to Die (Women's Murder Club #1),53.98,5694.89,1
9,44,Tipping the Velvet,53.74,5669.57,1


In [40]:
query1_df.to_csv(
    "data_pipeline/output/query_1.csv",
    index=False
)

print("Query 1 saved!")

Query 1 saved!


In [41]:
query2 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

query2_df = pd.read_sql(
    query2,
    conn
)

display(
    query2_df
)

,title,price_gbp,price_inr,rating
0,Boar Island (Anna Pigeon #19),59.48,6275.14,3
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4
2,A Year in Provence (Provence #1),56.88,6000.84,4
3,The Past Never Ends,56.50,5960.75,4
4,The Last Painting of Sara de Vos,55.55,5860.52,2
5,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5
6,Murder at the 42nd Street Library (Raymond Amb...,54.36,5734.98,4
7,The Last Mile (Amos Decker #2),54.21,5719.16,2
8,1st to Die (Women's Murder Club #1),53.98,5694.89,1
9,Tipping the Velvet,53.74,5669.57,1


In [42]:
query2_df.to_csv(
    "data_pipeline/output/query_2.csv",
    index=False
)

print("Query 2 saved!")

Query 2 saved!


In [43]:
query3 = """
SELECT DISTINCT
    rating
FROM books
ORDER BY rating;
"""

query3_df = pd.read_sql(
    query3,
    conn
)

display(
    query3_df
)

,rating
0,1
1,2
2,3
3,4
4,5


In [44]:
query3_df.to_csv(
    "data_pipeline/output/query_3.csv",
    index=False
)

print("Query 3 saved!")

Query 3 saved!


In [45]:
query4 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp;
"""

query4_df = pd.read_sql(
    query4,
    conn
)

display(
    query4_df
)

,title,price_gbp,price_inr,rating
0,Blood Defense (Samantha Brinkman #1),20.30,2141.65,3
1,"Love, Lies and Spies",20.55,2168.02,2
2,Between Shades of Gray,20.79,2193.34,5
3,Delivering the Truth (Quaker Midwife Mystery #1),20.89,2203.90,4
4,Voyager (Outlander #3),21.07,2222.89,5
5,The Silkworm (Cormoran Strike #2),23.05,2431.78,5
6,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1
7,Career of Evil (Cormoran Strike #3),24.72,2607.96,2
8,The Mysterious Affair at Styles (Hercule Poiro...,24.80,2616.40,4
9,What Happened on Beale Street (Secrets of the ...,25.37,2676.54,5


In [46]:
query4_df.to_csv(
    "data_pipeline/output/query_4.csv",
    index=False
)

print("Query 4 saved!")

Query 4 saved!


In [47]:
query5 = """
SELECT
    title,
    price_gbp,
    price_inr,
    rating
FROM books
WHERE rating IN (4, 5)
ORDER BY
    rating DESC,
    price_gbp DESC;
"""

query5_df = pd.read_sql(
    query5,
    conn
)

display(
    query5_df
)

,title,price_gbp,price_inr,rating
0,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5
1,The Bachelor Girl's Guide to Murder (Herringfo...,52.30,5517.65,5
2,A Time of Torment (Charlie Parker #14),48.35,5100.92,5
3,While You Were Mine,41.32,4359.26,5
4,The Red Tent,35.66,3762.13,5
5,Mrs. Houdini,30.25,3191.38,5
6,The Passion of Dolssa,28.32,2987.76,5
7,"1,000 Places to See Before You Die",26.08,2751.44,5
8,What Happened on Beale Street (Secrets of the ...,25.37,2676.54,5
9,The Silkworm (Cormoran Strike #2),23.05,2431.78,5


In [48]:
query5_df.to_csv(
    "data_pipeline/output/query_5.csv",
    index=False
)

print("Query 5 saved!")

Query 5 saved!


In [49]:
join_query = """
SELECT
    b.book_id,
    b.title,
    b.price_gbp,
    b.price_inr,
    b.rating,
    b.in_stock,
    c.category_id,
    c.category_name
FROM books AS b
INNER JOIN categories AS c
    ON b.category_id = c.category_id
ORDER BY
    c.category_name ASC,
    b.rating DESC,
    b.title ASC;
"""

sql_join_df = pd.read_sql(
    join_query,
    conn
)

display(
    sql_join_df.head(20)
)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5,1,1,Historical Fiction
1,69,A Spy's Devotion (The Regency Spies of London #1),16.97,1790.33,5,1,1,Historical Fiction
2,64,Between Shades of Gray,20.79,2193.34,5,1,1,Historical Fiction
3,48,Mrs. Houdini,30.25,3191.38,5,1,1,Historical Fiction
4,57,The Passion of Dolssa,28.32,2987.76,5,1,1,Historical Fiction
5,60,The Red Tent,35.66,3762.13,5,1,1,Historical Fiction
6,59,Voyager (Outlander #3),21.07,2222.89,5,1,1,Historical Fiction
7,65,While You Were Mine,41.32,4359.26,5,1,1,Historical Fiction
8,52,A Paris Apartment,39.01,4115.55,4,1,1,Historical Fiction
9,68,Lost Among the Living,27.70,2922.35,4,1,1,Historical Fiction


In [50]:
sql_join_df.to_csv(
    "data_pipeline/output/query_6_join.csv",
    index=False
)

print(
    "SQL JOIN result saved!"
)

SQL JOIN result saved!


In [51]:
read_sql_result_1 = pd.read_sql(
    query2,
    conn
)

print(
    "Result 1 using pd.read_sql():"
)

display(
    read_sql_result_1
)

Result 1 using pd.read_sql():


,title,price_gbp,price_inr,rating
0,Boar Island (Anna Pigeon #19),59.48,6275.14,3
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,57.70,6087.35,4
2,A Year in Provence (Provence #1),56.88,6000.84,4
3,The Past Never Ends,56.50,5960.75,4
4,The Last Painting of Sara de Vos,55.55,5860.52,2
5,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5
6,Murder at the 42nd Street Library (Raymond Amb...,54.36,5734.98,4
7,The Last Mile (Amos Decker #2),54.21,5719.16,2
8,1st to Die (Women's Murder Club #1),53.98,5694.89,1
9,Tipping the Velvet,53.74,5669.57,1


In [52]:
read_sql_result_2 = pd.read_sql(
    join_query,
    conn
)

print(
    "Result 2 using pd.read_sql():"
)

display(
    read_sql_result_2.head(20)
)

Result 2 using pd.read_sql():


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5,1,1,Historical Fiction
1,69,A Spy's Devotion (The Regency Spies of London #1),16.97,1790.33,5,1,1,Historical Fiction
2,64,Between Shades of Gray,20.79,2193.34,5,1,1,Historical Fiction
3,48,Mrs. Houdini,30.25,3191.38,5,1,1,Historical Fiction
4,57,The Passion of Dolssa,28.32,2987.76,5,1,1,Historical Fiction
5,60,The Red Tent,35.66,3762.13,5,1,1,Historical Fiction
6,59,Voyager (Outlander #3),21.07,2222.89,5,1,1,Historical Fiction
7,65,While You Were Mine,41.32,4359.26,5,1,1,Historical Fiction
8,52,A Paris Apartment,39.01,4115.55,4,1,1,Historical Fiction
9,68,Lost Among the Living,27.70,2922.35,4,1,1,Historical Fiction


In [54]:
books_db_df = pd.read_sql(
    """
    SELECT *
    FROM books
    """,
    conn
)

categories_db_df = pd.read_sql(
    """
    SELECT *
    FROM categories
    """,
    conn
)

print("Books table:")

display(
    books_db_df.head()
)

print("Categories table:")

display(
    categories_db_df.head()
)

Books table:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.44,2,1,3
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,3
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,3
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,3
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,3


Categories table:


,category_id,category_name
0,1,Historical Fiction
1,2,Mystery
2,3,Travel


In [55]:
pandas_join_df = pd.merge(
    books_db_df,
    categories_db_df,
    on="category_id",
    how="inner"
)

pandas_join_df = pandas_join_df[
    [
        "book_id",
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category_id",
        "category_name"
    ]
]

pandas_join_df = (
    pandas_join_df
    .sort_values(
        by=[
            "category_name",
            "rating",
            "title"
        ],
        ascending=[
            True,
            False,
            True
        ]
    )
    .reset_index(drop=True)
)

display(
    pandas_join_df.head(20)
)

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5,1,1,Historical Fiction
1,69,A Spy's Devotion (The Regency Spies of London #1),16.97,1790.33,5,1,1,Historical Fiction
2,64,Between Shades of Gray,20.79,2193.34,5,1,1,Historical Fiction
3,48,Mrs. Houdini,30.25,3191.38,5,1,1,Historical Fiction
4,57,The Passion of Dolssa,28.32,2987.76,5,1,1,Historical Fiction
5,60,The Red Tent,35.66,3762.13,5,1,1,Historical Fiction
6,59,Voyager (Outlander #3),21.07,2222.89,5,1,1,Historical Fiction
7,65,While You Were Mine,41.32,4359.26,5,1,1,Historical Fiction
8,52,A Paris Apartment,39.01,4115.55,4,1,1,Historical Fiction
9,68,Lost Among the Living,27.70,2922.35,4,1,1,Historical Fiction


In [56]:
sql_comparison = (
    sql_join_df
    .reset_index(drop=True)
)

pandas_comparison = (
    pandas_join_df
    .reset_index(drop=True)
)

print(
    "SQL JOIN shape:",
    sql_comparison.shape
)

print(
    "Pandas merge shape:",
    pandas_comparison.shape
)

SQL JOIN shape: (69, 8)
Pandas merge shape: (69, 8)


In [57]:
comparison_matches = (
    sql_comparison.equals(
        pandas_comparison
    )
)

print(
    "Do SQL JOIN and pandas merge match?"
)

print(
    comparison_matches
)

Do SQL JOIN and pandas merge match?
True


In [58]:
comparison_df = pd.concat(
    [
        sql_comparison
        .head(10)
        .add_prefix("SQL_"),

        pandas_comparison
        .head(10)
        .add_prefix("PANDAS_")
    ],
    axis=1
)

display(
    comparison_df
)

,SQL_book_id,SQL_title,SQL_price_gbp,SQL_price_inr,SQL_rating,SQL_in_stock,SQL_category_id,SQL_category_name,PANDAS_book_id,PANDAS_title,PANDAS_price_gbp,PANDAS_price_inr,PANDAS_rating,PANDAS_in_stock,PANDAS_category_id,PANDAS_category_name
0,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5,1,1,Historical Fiction,46,A Flight of Arrows (The Pathfinders #2),55.53,5858.42,5,1,1,Historical Fiction
1,69,A Spy's Devotion (The Regency Spies of London #1),16.97,1790.33,5,1,1,Historical Fiction,69,A Spy's Devotion (The Regency Spies of London #1),16.97,1790.33,5,1,1,Historical Fiction
2,64,Between Shades of Gray,20.79,2193.34,5,1,1,Historical Fiction,64,Between Shades of Gray,20.79,2193.34,5,1,1,Historical Fiction
3,48,Mrs. Houdini,30.25,3191.38,5,1,1,Historical Fiction,48,Mrs. Houdini,30.25,3191.38,5,1,1,Historical Fiction
4,57,The Passion of Dolssa,28.32,2987.76,5,1,1,Historical Fiction,57,The Passion of Dolssa,28.32,2987.76,5,1,1,Historical Fiction
5,60,The Red Tent,35.66,3762.13,5,1,1,Historical Fiction,60,The Red Tent,35.66,3762.13,5,1,1,Historical Fiction
6,59,Voyager (Outlander #3),21.07,2222.89,5,1,1,Historical Fiction,59,Voyager (Outlander #3),21.07,2222.89,5,1,1,Historical Fiction
7,65,While You Were Mine,41.32,4359.26,5,1,1,Historical Fiction,65,While You Were Mine,41.32,4359.26,5,1,1,Historical Fiction
8,52,A Paris Apartment,39.01,4115.55,4,1,1,Historical Fiction,52,A Paris Apartment,39.01,4115.55,4,1,1,Historical Fiction
9,68,Lost Among the Living,27.70,2922.35,4,1,1,Historical Fiction,68,Lost Among the Living,27.70,2922.35,4,1,1,Historical Fiction


In [59]:
comparison_df.to_csv(
    "data_pipeline/output/join_comparison.csv",
    index=False
)

print(
    "JOIN comparison saved!"
)

JOIN comparison saved!


In [61]:
all_queries = {
    "Query 1 - SELECT and WHERE": query1,
    "Query 2 - ORDER BY and LIMIT": query2,
    "Query 3 - DISTINCT": query3,
    "Query 4 - BETWEEN": query4,
    "Query 5 - IN": query5,
    "Query 6 - JOIN": join_query
}

with open(
    "data_pipeline/output/sql_queries.txt",
    "w",
    encoding="utf-8"
) as file:

    for name, query in all_queries.items():

        file.write(
            "=" * 70 + "\n"
        )

        file.write(
            name + "\n"
        )

        file.write(
            "=" * 70 + "\n"
        )

        file.write(
            query.strip() + "\n\n"
        )

print(
    "All SQL queries saved!"
)

All SQL queries saved!


In [62]:
query_outputs = {
    "Query 1 Output": query1_df,
    "Query 2 Output": query2_df,
    "Query 3 Output": query3_df,
    "Query 4 Output": query4_df,
    "Query 5 Output": query5_df,
    "Query 6 JOIN Output": sql_join_df
}

with open(
    "data_pipeline/output/sql_query_outputs.txt",
    "w",
    encoding="utf-8"
) as file:

    for name, dataframe in query_outputs.items():

        file.write(
            "\n" + "=" * 80 + "\n"
        )

        file.write(
            name + "\n"
        )

        file.write(
            "=" * 80 + "\n"
        )

        file.write(
            dataframe.to_string(index=False)
        )

        file.write("\n")

print(
    "SQL query outputs saved!"
)

SQL query outputs saved!


In [63]:
assert len(final_df) >= 60

assert (
    final_df["category"]
    .nunique()
    >= 3
)

assert pd.api.types.is_float_dtype(
    final_df["price_gbp"]
)

assert pd.api.types.is_integer_dtype(
    final_df["rating"]
)

assert pd.api.types.is_bool_dtype(
    final_df["in_stock"]
)

assert pd.api.types.is_float_dtype(
    final_df["price_inr"]
)

assert (
    final_df["rating"]
    .between(1, 5)
    .all()
)

expected_inr = (
    final_df["price_gbp"]
    * 105.50
).round(2)

assert final_df["price_inr"].equals(
    expected_inr
)

assert sql_comparison.equals(
    pandas_comparison
)

print("=" * 60)

print(
    "ALL ACCEPTANCE CHECKS PASSED"
)

print("=" * 60)

print(
    "Total books:",
    len(final_df)
)

print(
    "Total categories:",
    final_df["category"]
    .nunique()
)

print(
    "GBP to INR rate: "
    "1 GBP = 105.50 INR"
)

print(
    "SQL JOIN == Pandas merge:",
    True
)

ALL ACCEPTANCE CHECKS PASSED
Total books: 69
Total categories: 3
GBP to INR rate: 1 GBP = 105.50 INR
SQL JOIN == Pandas merge: True


In [64]:
print(
    "FINAL CLEANED DATASET"
)

display(
    final_df
)

FINAL CLEANED DATASET


,title,price_gbp,price_inr,rating,in_stock,category
0,It's Only the Himalayas,45.17,4765.44,2,True,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,True,Travel
2,See America: A Celebration of Our National Par...,48.87,5155.78,3,True,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,True,Travel
4,Under the Tuscan Sun,37.33,3938.31,3,True,Travel
...,...,...,...,...,...,...
64,While You Were Mine,41.32,4359.26,5,True,Historical Fiction
65,The Secret Healer,34.56,3646.08,3,True,Historical Fiction
66,Starlark,25.83,2725.06,3,True,Historical Fiction
67,Lost Among the Living,27.70,2922.35,4,True,Historical Fiction


In [65]:
print("CATEGORIES TABLE")

display(
    pd.read_sql(
        "SELECT * FROM categories",
        conn
    )
)

print("\nBOOKS TABLE")

display(
    pd.read_sql(
        "SELECT * FROM books",
        conn
    ).head(20)
)

CATEGORIES TABLE


,category_id,category_name
0,1,Historical Fiction
1,2,Mystery
2,3,Travel



BOOKS TABLE


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,It's Only the Himalayas,45.17,4765.44,2,1,3
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,49.43,5214.86,4,1,3
2,3,See America: A Celebration of Our National Par...,48.87,5155.78,3,1,3
3,4,Vagabonding: An Uncommon Guide to the Art of L...,36.94,3897.17,2,1,3
4,5,Under the Tuscan Sun,37.33,3938.31,3,1,3
5,6,A Summer In Europe,44.34,4677.87,2,1,3
6,7,The Great Railway Bazaar,30.54,3221.97,1,1,3
7,8,A Year in Provence (Provence #1),56.88,6000.84,4,1,3
8,9,The Road to Little Dribbling: Adventures of an...,23.21,2448.66,1,1,3
9,10,Neither Here nor There: Travels in Europe,38.95,4109.23,3,1,3


In [67]:
print(
    "Files generated by the project:"
)

for root, dirs, files in os.walk(
    "data_pipeline"
):

    level = root.replace(
        "data_pipeline",
        ""
    ).count(os.sep)

    indent = "    " * level

    print(
        f"{indent}{os.path.basename(root)}/"
    )

    for filename in files:

        print(
            f"{indent}    {filename}"
        )

Files generated by the project:
data_pipeline/
    books.db
    data/
        cleaned_books.csv
        raw_books.csv
    output/
        join_comparison.csv
        query_1.csv
        query_2.csv
        query_3.csv
        query_4.csv
        query_5.csv
        query_6_join.csv
        sql_queries.txt
        sql_query_outputs.txt


In [68]:
conn.close()

print(
    "Database connection closed successfully!"
)

Database connection closed successfully!
